# Reproduce the site-level wastewater resistome analysis (Google Colab version)

Secondary analysis of the public processed tables from Singh et al., *Nature Communications* (2026), doi:10.1038/s41467-026-70702-x.

**How to run:** use *Runtime > Run all*. When asked, upload `github_repo.zip` (or clone the repository once it is public). The notebook downloads the public data archive itself, checks its checksum, runs steps 2 to 6, and compares the regenerated tables with the committed ones. Expected run time: a few minutes.

**Status:** research code accompanying a manuscript draft; not peer reviewed. Analysis code was drafted with AI assistance (Claude, Anthropic) [authors: confirm wording and that you have reviewed the code].

## 1. Get the repository

In [ ]:
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if Path("scripts/step2_sample_level_cpm.py").exists():
    print("Already inside the repository:", Path.cwd())
elif IN_COLAB:
    from google.colab import files
    print("Upload github_repo.zip when prompted.")
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z: z.extractall("/content")
    os.chdir("/content/github_repo")
    print("Working directory:", Path.cwd())
else:
    raise SystemExit("Run this notebook from the repository root, or open it in Google Colab.")

## 2. Check the software versions
The committed tables were produced with NumPy 2.4.4, pandas 3.0.2, SciPy 1.17.1 and Matplotlib 3.10.8. Colab may have different versions; the final check cell will tell you if that changes any result. If it does, run the optional pinned install below, then *Runtime > Restart session* and run all cells again.

In [ ]:
import numpy, pandas, scipy, matplotlib
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)
# Optional: uncomment to install the exact versions used for the committed results, then restart the session.
# !pip install -q numpy==2.4.4 pandas==3.0.2 scipy==1.17.1 matplotlib==3.10.8

## 3. Download the public data
Source: Zenodo record 10.5281/zenodo.17590646 (original authors' archive; confirm the record and its licence before reuse). The archive checksum is verified. If the download fails, download `data.tar.xz` manually from the record, upload it to `data/raw/`, and run this cell again.

In [ ]:
import hashlib, tarfile, urllib.request
RAW = Path("data/raw"); RAW.mkdir(parents=True, exist_ok=True)
ARCHIVE = RAW / "data.tar.xz"
URL = "https://zenodo.org/records/17590646/files/data.tar.xz?download=1"
EXPECTED_MD5 = "dd9e8bfaaad2aa5b33c81c7fd1a8b5bc"

def find_data_dir():
    hits = [p.parent for p in RAW.rglob("meta.tsv")]
    return hits[0] if hits else None

if find_data_dir() is None:
    if not ARCHIVE.exists():
        print("Downloading archive ...")
        urllib.request.urlretrieve(URL, ARCHIVE)
    md5 = hashlib.md5(ARCHIVE.read_bytes()).hexdigest()
    if md5 != EXPECTED_MD5:
        raise RuntimeError(f"Checksum mismatch: {md5} (expected {EXPECTED_MD5}). Re-download the archive and check the Zenodo record.")
    print("Checksum OK; extracting ...")
    with tarfile.open(ARCHIVE) as t: t.extractall(RAW)

DATA_DIR = str(find_data_dir()) + "/"
need = ["meta.tsv", "rgi_bowtie.tsv", "bacterial_read_df.tsv"]
missing = [f for f in need if not Path(DATA_DIR + f).exists()]
assert not missing, f"Missing {missing} in {DATA_DIR}"
print("Data folder OK:", DATA_DIR)

## 4. Run the analysis (steps 2 to 6)

In [ ]:
import time
RESULTS = "results/"; Path(RESULTS + "figures").mkdir(parents=True, exist_ok=True)
SITE_MAP = "tables/master_sample_site_map.csv"   # sample -> site key (from NCBI BioProject PRJNA1415085)
CPM = RESULTS + "step2_sample_level_cpm.csv"

def run(script, *args):
    t = time.time()
    r = subprocess.run([sys.executable, script, *args], capture_output=True, text=True)
    print(r.stdout[-800:])
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise RuntimeError(f"{script} failed")
    print(f"{script}: done in {time.time()-t:.0f} s\n")

run("scripts/step2_sample_level_cpm.py", DATA_DIR, SITE_MAP, RESULTS)      # sample-level CPM and detection
run("scripts/step3_primary_analysis.py", CPM, RESULTS)                     # analyses A, B, C
run("scripts/step4_sensitivity.py", DATA_DIR, SITE_MAP, RESULTS)           # sensitivity and leave-one-site-out
run("scripts/step5_figures.py", CPM, RESULTS, RESULTS + "figures/")        # figures 1, 2, 4, 5
run("scripts/step5b_composition_figure.py", CPM, RESULTS + "figures/")     # figure 3
run("scripts/step6_audit_extras.py", CPM, RESULTS)                         # post hoc analyses added after review

## 5. Check: do the regenerated tables match the committed ones?
`OK` means identical to within 1e-9. `CHECK` means a difference: first try the pinned install in section 2, then investigate before trusting the reproduction.

In [ ]:
import pandas as pd, numpy as np
files = ["step3_A_permanova_by_city.csv", "step3_B_summary_by_city.csv", "step3_C_two_site_stability.csv",
         "step4_A_sensitivity.csv", "step4_LOSO.csv",
         "step6_C_exact_enumeration.csv", "step6_A_site_given_month.csv", "step6_Mumbai_exclusion.csv"]
for f in files:
    new, old = pd.read_csv(RESULTS + f), pd.read_csv("tables/" + f)
    num = [c for c in old.columns if pd.api.types.is_numeric_dtype(old[c])]
    same = new.shape == old.shape
    diff = float(np.nanmax(np.abs(new[num].values - old[num].values))) if same else float("nan")
    print(f"{f:36s} shape match: {same}   max abs difference: {diff:.2e}   {'OK' if same and diff < 1e-9 else 'CHECK'}")

## 6. Look at the figures

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
for f in ["Fig1_sampling_coverage", "Fig2_site_ordination", "Fig3_site_composition", "Fig4_aggregation_comparison", "Fig5_two_site_stability"]:
    plt.figure(figsize=(8, 6)); plt.imshow(mpimg.imread(f"{RESULTS}figures/{f}.png")); plt.axis("off"); plt.title(f); plt.show()

## 7. Download the results (optional)

In [ ]:
shutil.make_archive("results_download", "zip", "results")
if IN_COLAB:
    from google.colab import files
    files.download("results_download.zip")
else:
    print("Created results_download.zip")